# For ddpm outpainting

In [1]:
n = 2
idx = (0, 0)

def update(idx, move):
    x, y = idx
    if move == 0:
        x += 0.5
    if move == 1:
        y += 0.5
    if move == 2:
        x -= 0.5
    if move == 3:
        y -= 0.5

    return (x, y)

def outpaint_scheduler(idx, n):
    for i in range(1,n+1,1):
        print("i",i)
        j = 0
        if i % 2 == 1:
            idx = update(idx, 0)
            #outpaint blocks (0,1) with idx -=(-0.5,0) 's (0,1)
            for j in range(i):
                idx = update(idx, 1)
                #if j == 0 outpaint 2 blocks else 1 blocks
            for j in range(i):
                idx = update(idx, 2)
        else:
      
            idx = update(idx, 2)
       
            for j in range(i):
                idx = update(idx, 3)
            for j in range(i):
                idx = update(idx, 0)
        print(idx)
    return idx


idx = outpaint_scheduler(idx, 3)
    # print(n," Final idx:", idx)


i 1
(0.0, 0.5)
i 2
(0.5, -0.5)
i 3
(-0.5, 1.0)


In [2]:
def outpaint_path(n, step=0.5, start=(0.0, 0.0)):
    x, y = start
    path = []
    def go(dx, dy, times):
        nonlocal x, y
        for _ in range(times):
            x += dx*step; y += dy*step
            path.append((x, y))
    for i in range(1, n+1):
        if i % 2 == 1:            # odd ring: R half, then U^i, L^i
            go(1, 0, 1)
            go(0, 1, i)
            go(-1, 0, i)
        else:                      # even ring: L half, then D^i, R^i
            go(-1, 0, 1)
            go(0, -1, i)
            go(1, 0, i)
    return path


In [3]:
import torch
import torch

def split2splitbig(split: torch.Tensor) -> torch.Tensor:
    """
    Expand an octant-coded grid into a dense grid with a singleton channel.

    Input:
        split: [B, 8, L, L, L]
            Channels 0..7 map to (e/o) along (z, y, x) respectively.

    Output:
        split_big: [B, 1, 2L, 2L, 2L]
    """
    assert split.ndim == 5, f"Expected [B, 8, L, L, L], got {split.shape}"
    B, C, L, H, W = split.shape
    assert C == 8, f"Channel dimension must be 8, got {C}"

    split_big = torch.full((B, 1, 2*L, 2*H, 2*W), -1, dtype=split.dtype, device=split.device)

    # Work on the single output channel
    sb = split_big[:, 0]

    # Fill each octant
    sb[:, 0::2, 0::2, 0::2] = split[:, 0]  # (e,e,e)
    sb[:, 0::2, 0::2, 1::2] = split[:, 1]  # (e,e,o)
    sb[:, 0::2, 1::2, 0::2] = split[:, 2]  # (e,o,e)
    sb[:, 0::2, 1::2, 1::2] = split[:, 3]  # (e,o,o)
    sb[:, 1::2, 0::2, 0::2] = split[:, 4]  # (o,e,e)
    sb[:, 1::2, 0::2, 1::2] = split[:, 5]  # (o,e,o)
    sb[:, 1::2, 1::2, 0::2] = split[:, 6]  # (o,o,e)
    sb[:, 1::2, 1::2, 1::2] = split[:, 7]  # (o,o,o)

    return split_big




import torch

def splitbig2split(split_big: torch.Tensor) -> torch.Tensor:
    """
    Collapse a dense voxel grid back into an octant-coded tensor.

    Accepts:
        [B, 1, 2L, 2L, 2L]  or  [B, 2L, 2L, 2L]
    Returns:
        [B, 8, L, L, L]
    """
    if split_big.ndim == 5:
        B, C, D, H, W = split_big.shape
        assert C == 1, f"Expected channel=1, got {C}"
        sb = split_big[:, 0]
    elif split_big.ndim == 4:
        B, D, H, W = split_big.shape
        sb = split_big
    else:
        raise AssertionError(f"Expected [B,1,2L,2L,2L] or [B,2L,2L,2L], got {split_big.shape}")

    assert D % 2 == 0 and H % 2 == 0 and W % 2 == 0, "Spatial dims must be even"
    L = D // 2

    out = torch.empty((B, 8, L, L, L), dtype=sb.dtype, device=sb.device)
    out[:, 0] = sb[:, 0::2, 0::2, 0::2]  # (e,e,e)
    out[:, 1] = sb[:, 0::2, 0::2, 1::2]  # (e,e,o)
    out[:, 2] = sb[:, 0::2, 1::2, 0::2]  # (e,o,e)
    out[:, 3] = sb[:, 0::2, 1::2, 1::2]  # (e,o,o)
    out[:, 4] = sb[:, 1::2, 0::2, 0::2]  # (o,e,e)
    out[:, 5] = sb[:, 1::2, 0::2, 1::2]  # (o,e,o)
    out[:, 6] = sb[:, 1::2, 1::2, 0::2]  # (o,o,e)
    out[:, 7] = sb[:, 1::2, 1::2, 1::2]  # (o,o,o)
    return out




In [4]:
@torch.no_grad()
def ddim_sample_logsnr_cosine_inpaint(z_T,z_gt, model,mask, doctree, S=50, jump_n_sample = 10,predict_mode="x0"):
    """
    DDIM sampling using cosine logSNR schedule and x₀ prediction.

    Args:
        z_T: Initial noise tensor [N, C]
        model: Trained diffusion model
        doctree: DualOctree structure
        S: Number of DDIM steps (e.g., 50)
        predict_mode: "x0" (only supported here)
    """
    assert predict_mode == "x0", "Only x0 mode is supported in this version."

    # === Step 1: Generate continuous t schedule [1.0, ..., 0.0]
    t_steps = torch.linspace(1., 0., S + 1, device=z_T.device)  # [S+1]
    t_pairs = list(zip(t_steps[:-1], t_steps[1:]))


    z_t = z_T
    for t, t_next in t_pairs:
        # Convert scalars to tensors
        for j in range(jump_n_sample): 
            t_tensor = t.view(1, 1)
            t_next_tensor = t_next.view(1, 1)

            logsnr_t = alpha_cosine_log_snr(t_tensor)
            logsnr_next = alpha_cosine_log_snr(t_next_tensor)

            alpha_t, sigma_t = log_snr_to_alpha_sigma(logsnr_t)
            alpha_next, sigma_next = log_snr_to_alpha_sigma(logsnr_next)
            # Inpaint
            noised_z_gt = (z_gt * alpha_next + sigma_next * torch.randn_like(z_gt))
            x0_pred = model(z_t, doctree=doctree, timesteps=t_tensor)  # Predict x0
            x0_pred = ( mask) * noised_z_gt + (1-mask) * x0_pred
            # DDIM update rule
            z_t = alpha_next * x0_pred + sigma_next * torch.randn_like(z_t)
            if j < jump_n_sample - 1 and t > 1:
                noise_back = torch.randn_like(z_t)
                # noised_data = alpha_t * noised_data + sigma_t * noise_back
                z_t = alpha_t * z_t + sigma_t * noise_back
    return z_t

In [5]:
import os

# Outdoor weights and the SemanticKITTI root are not in the repo. Point these at
# wherever you unpacked them; the defaults assume they sit under the repo.
#   export OCTREE_DIFF_KITTI_WEIGHTS=/path/to/weights
#   export OCTREE_DIFF_KITTI_DATA=/path/to/semantic-kitti
KITTI_WEIGHTS = os.environ.get("OCTREE_DIFF_KITTI_WEIGHTS", "weights/kitti")
KITTI_DATA = os.environ.get("OCTREE_DIFF_KITTI_DATA", "data/kitti")

from octree_diff.viz.open3d_viewers import visualize_kitti_instance, visualize_structure

from octree_diff.diffusion.util_sample_stuff import *
from octree_diff.octree.util_octree_stuff import *
from octree_diff.octree.util_dualoctree import *

In [6]:
from octree_diff.models.outdoor.graph_unet_lr import UNet3DModel
batch_size = 1
in_channels = 1

full_depth = 4  # dummy value, used only in forward_as_middle
model_channels = 64
out_channels = 1



structure_model = UNet3DModel(
    full_depth=full_depth,
    in_split_channels=in_channels,
    model_channels=model_channels,
    out_split_channels=out_channels,
    channel_mult=(1, 2, 4,8),
    attention_resolutions=[2, 4, 8],  # example downsample scales
    dims=3,  # 3D model
    num_heads=4,
    timestep_repr = "ddpm",
    T = 1000,
).cuda()


In [7]:
import os
import torch

def save_checkpoint(path, model, optimizer=None, scheduler=None, ema=None,
                    epoch=0, global_step=0, extra: dict | None=None):
    ckpt = {
        "model": model.state_dict(),
        "epoch": epoch,
        "global_step": global_step,
    }
    if optimizer is not None:
        ckpt["optimizer"] = optimizer.state_dict()
    if scheduler is not None:
        ckpt["scheduler"] = scheduler.state_dict()
    if ema is not None:
        # store EMA shadow weights
        ckpt["ema_shadow"] = {k: v.clone().cpu() for k, v in ema.shadow.items()}
        ckpt["ema_decay"] = ema.decay
    if extra:
        ckpt["extra"] = extra
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save(ckpt, path)
    print(f"[+] Saved checkpoint to {path}")

def load_checkpoint(path, model, optimizer=None, scheduler=None, ema=None,
                    map_location="cpu", strict=True):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model"], strict=strict)
    print(f"[+] Loaded model weights from {path}")

    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
        print("[+] Loaded optimizer state")
    if scheduler is not None and "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
        print("[+] Loaded scheduler state")
    if ema is not None and "ema_shadow" in ckpt:
        # ensure keys match; if model changed, set strict=False earlier
        ema.shadow = {k: v.to(next(model.parameters()).device) for k, v in ckpt["ema_shadow"].items()}
        if "ema_decay" in ckpt:
            ema.decay = ckpt["ema_decay"]
        print("[+] Loaded EMA shadow")

    epoch = ckpt.get("epoch", 0)
    global_step = ckpt.get("global_step", 0)
    extra = ckpt.get("extra", {})
    return epoch, global_step, extra

epoch, global_step, _ = load_checkpoint(
    "checkpoints/structure_ddpm_ep110.pt",
    model=structure_model,
    map_location="cuda",     # or your device
    strict=True,
)



/tmp/ipykernel_18767/362956149.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=map_location)


[+] Loaded model weights from checkpoints/structure_ddpm_ep110.pt


In [8]:
import torch
import torch.nn.functional as F

def build_ddpm_sampling_buffers(T, device, beta_start=1e-4, beta_end=2e-2):
    """Extend your linear schedule with helpful arrays for sampling."""
    betas = torch.linspace(beta_start, beta_end, T, dtype=torch.float32, device=device)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = torch.cat([torch.ones(1, device=device), alphas_cumprod[:-1]], dim=0)

    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
    sqrt_recip_alphas = torch.sqrt(1.0 / alphas)

    # posterior q(x_{t-1}|x_t, x0)
    posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
    # clip log var for t=0 (same trick as OpenAI / LDM)
    posterior_log_variance_clipped = torch.log(torch.clamp(posterior_variance, min=1e-20))
    posterior_mean_coef1 = betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod)
    posterior_mean_coef2 = (1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod)

    sched = {
        "betas": betas,
        "alphas": alphas,
        "alphas_cumprod": alphas_cumprod,
        "alphas_cumprod_prev": alphas_cumprod_prev,
        "sqrt_alphas_cumprod": sqrt_alphas_cumprod,
        "sqrt_one_minus_alphas_cumprod": sqrt_one_minus_alphas_cumprod,
        "sqrt_recip_alphas": sqrt_recip_alphas,
        "posterior_variance": posterior_variance,
        "posterior_log_variance_clipped": posterior_log_variance_clipped,
        "posterior_mean_coef1": posterior_mean_coef1,
        "posterior_mean_coef2": posterior_mean_coef2,
    }
    return sched

def _extract(a, t, x_shape):
    """
    a: [T], t: [B], x_shape: e.g. (B,C,D,H,W)
    returns [B,1,1,1,1] broadcastable to x
    """
    out = a.gather(0, t)
    return out.view(-1, *([1] * (len(x_shape) - 1)))

@torch.no_grad()
def p_mean_variance_eps(model, x_t, t, sched):
    """
    Predict posterior mean & variance using ε-prediction.
    Returns: mean, log_variance, x0_pred
    """
    # eps_theta
    eps_theta = model(x_t, timesteps=t)

    sqrt_alphabar_t = _extract(sched["sqrt_alphas_cumprod"], t, x_t.shape)
    sqrt_one_minus_alphabar_t = _extract(sched["sqrt_one_minus_alphas_cumprod"], t, x_t.shape)

    # x0 prediction from eps
    x0_pred = (x_t - sqrt_one_minus_alphabar_t * eps_theta) / (sqrt_alphabar_t + 1e-8)

    # posterior mean
    c1 = _extract(sched["posterior_mean_coef1"], t, x_t.shape)
    c2 = _extract(sched["posterior_mean_coef2"], t, x_t.shape)
    mean = c1 * x0_pred + c2 * x_t

    # posterior variance (log)
    log_variance = _extract(sched["posterior_log_variance_clipped"], t, x_t.shape)
    return mean, log_variance, x0_pred

@torch.no_grad()
def sample_ddpm(model, sched, shape, T=1000, device="cuda", x_T=None):
    """
    Classic ancestral DDPM sampling (full 1000 steps).
    - model: ε-pred UNet that accepts integer t in [0..T-1]
    - sched: dict from build_ddpm_sampling_buffers(...)
    - shape: (B,C,D,H,W)
    - x_T: optional starting noise [B,C,D,H,W]
    """
    if x_T is None:
        x_t = torch.randn(shape, device=device)
    else:
        x_t = x_T.to(device)

    model.eval()
    for t_int in reversed(range(T)):
        t = torch.full((shape[0],), t_int, device=device, dtype=torch.long)

        mean, log_var, x0_pred = p_mean_variance_eps(model, x_t, t, sched)

        if t_int > 0:
            noise = torch.randn_like(x_t)
            x_t = mean + torch.exp(0.5 * log_var) * noise
        else:
            # final step: return the x_0 estimate (mean is fine too)
            x_t = x0_pred
    return x_t
import torch

def split2splitbig(split: torch.Tensor) -> torch.Tensor:
    """
    Expand an octant-coded grid into a dense grid with a singleton channel.

    Input:
        split: [B, 8, L, L, L]
            Channels 0..7 map to (e/o) along (z, y, x) respectively.

    Output:
        split_big: [B, 1, 2L, 2L, 2L]
    """
    assert split.ndim == 5, f"Expected [B, 8, L, L, L], got {split.shape}"
    B, C, L, H, W = split.shape
    assert C == 8, f"Channel dimension must be 8, got {C}"

    split_big = torch.full((B, 1, 2*L, 2*H, 2*W), -1, dtype=split.dtype, device=split.device)

    # Work on the single output channel
    sb = split_big[:, 0]

    # Fill each octant
    sb[:, 0::2, 0::2, 0::2] = split[:, 0]  # (e,e,e)
    sb[:, 0::2, 0::2, 1::2] = split[:, 1]  # (e,e,o)
    sb[:, 0::2, 1::2, 0::2] = split[:, 2]  # (e,o,e)
    sb[:, 0::2, 1::2, 1::2] = split[:, 3]  # (e,o,o)
    sb[:, 1::2, 0::2, 0::2] = split[:, 4]  # (o,e,e)
    sb[:, 1::2, 0::2, 1::2] = split[:, 5]  # (o,e,o)
    sb[:, 1::2, 1::2, 0::2] = split[:, 6]  # (o,o,e)
    sb[:, 1::2, 1::2, 1::2] = split[:, 7]  # (o,o,o)

    return split_big




import torch

def splitbig2split(split_big: torch.Tensor) -> torch.Tensor:
    """
    Collapse a dense voxel grid back into an octant-coded tensor.

    Accepts:
        [B, 1, 2L, 2L, 2L]  or  [B, 2L, 2L, 2L]
    Returns:
        [B, 8, L, L, L]
    """
    if split_big.ndim == 5:
        B, C, D, H, W = split_big.shape
        assert C == 1, f"Expected channel=1, got {C}"
        sb = split_big[:, 0]
    elif split_big.ndim == 4:
        B, D, H, W = split_big.shape
        sb = split_big
    else:
        raise AssertionError(f"Expected [B,1,2L,2L,2L] or [B,2L,2L,2L], got {split_big.shape}")

    assert D % 2 == 0 and H % 2 == 0 and W % 2 == 0, "Spatial dims must be even"
    L = D // 2

    out = torch.empty((B, 8, L, L, L), dtype=sb.dtype, device=sb.device)
    out[:, 0] = sb[:, 0::2, 0::2, 0::2]  # (e,e,e)
    out[:, 1] = sb[:, 0::2, 0::2, 1::2]  # (e,e,o)
    out[:, 2] = sb[:, 0::2, 1::2, 0::2]  # (e,o,e)
    out[:, 3] = sb[:, 0::2, 1::2, 1::2]  # (e,o,o)
    out[:, 4] = sb[:, 1::2, 0::2, 0::2]  # (o,e,e)
    out[:, 5] = sb[:, 1::2, 0::2, 1::2]  # (o,e,o)
    out[:, 6] = sb[:, 1::2, 1::2, 0::2]  # (o,o,e)
    out[:, 7] = sb[:, 1::2, 1::2, 1::2]  # (o,o,o)
    return out


In [9]:
device = next(structure_model.parameters()).device
sched = build_ddpm_sampling_buffers(T=1000, device=device, beta_start=1e-4, beta_end=2e-2)
with torch.inference_mode():
    structure_model.eval()
    samples = sample_ddpm(structure_model, sched, shape=(1, 1,32,32,16), T=1000, device=device)


In [10]:
# visualize_kitti_instance(sample_32[0][0])
from octree_diff.octree.util_octree_stuff import *
samples_clone = samples.clone()
samples_clone[samples_clone>0] = 1
samples_clone[samples_clone<0] = 0
visualize_kitti_instance(samples_clone[0][0])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [11]:
z_T = torch.randn(1, 1, 32, 32, 16).to('cuda')  # starting noise
# sample = ddim_sample_structure_logsnr_cosine(z_T, structure_model,  S=100)
# sample[sample>0] = 1
# sample[sample<0] = 0
sample_32= torch.zeros(1,1,32,32,32).cuda()
sample_32[:,:,:,:,0:16] = samples
# visualize_kitti_instance(sample[0][0])
split_recon = splitbig2split(sample_32)

# filled = fill_ground_levels(sample_32[0], z_levels)

In [12]:
from octree_diff.models.semantic.graph_sem_vae import GraphVAE
DEPTH_STOP = 6
LATENT_DIM = 8
FULL_DEPTH = 3
vae = GraphVAE(
    depth=6,
    channel_in=32,     
    nout=2,
    full_depth=FULL_DEPTH,
    depth_stop=DEPTH_STOP ,
    depth_out=6,
    latent_dim=LATENT_DIM,  
    num_classes=21,# latent dimension
    resblk_num=1,
    patch_size=4,   # outdoor uses 4x4 patches; the default 2 will not load these weights
    
).cuda()

vae.load_state_dict(torch.load(f"{KITTI_WEIGHTS}/vae_fdepth6_ldim8_8.pt"),vae.parameters())

vae.eval() 

import torch
from octree_diff.models.semantic.dual_octree import DualOctree
from octree_diff.models.semantic.graph_unet_hr import UNet3DModel

# === 1. Hyperparams ===
in_channels = LATENT_DIM        # dimension of latent z
model_channels = 64

lr_model_channels = 128
out_channels = LATENT_DIM      # same as input if predicting noise
image_size = 16
depth = DEPTH_STOP
full_depth = FULL_DEPTH
channel_mult = [2,4]
num_res_blocks = [2, 2, 2]



# === 4. Init and forward model ===
unet_model = UNet3DModel(
    image_size=image_size,
    input_depth=depth,
    full_depth=full_depth,
    in_channels=in_channels,
    model_channels=model_channels,
    lr_model_channels=lr_model_channels,
    out_channels=out_channels,
    num_res_blocks=num_res_blocks,
    dropout=0.1,
    channel_mult=channel_mult,
    use_checkpoint=False,
).cuda()

unet_model.load_state_dict(torch.load(f"{KITTI_WEIGHTS}/Model_is_good_8_6.pt"),unet_model.parameters())
unet_model.eval()


/tmp/ipykernel_18767/2793336102.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vae.load_state_dict(torch.load("vae_fdepth6_ldim8_8.pt"),vae.parameters())
/tmp/ipykerne

UNet3DModel(
  (time_embed): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): SiLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
  )
  (input_blocks): ModuleList(
    (0): GraphConv(channel_in=8, channel_out=64, n_edge_type=7, avg_degree=7, n_node_type=5)
    (1): GraphResBlockEmbed(
      (block1_norm): DualOctreeGroupNorm(in_channels=64, group=32, nempty=False)
      (silu): SiLU()
      (conv1): GraphConv(channel_in=64, channel_out=128, n_edge_type=7, avg_degree=7, n_node_type=5)
      (emb_layers): Sequential(
        (0): SiLU()
        (1): Linear(in_features=256, out_features=128, bias=True)
      )
      (block2_norm): DualOctreeGroupNorm(in_channels=128, group=32, nempty=False)
      (dropout): Dropout(p=0.1, inplace=False)
      (conv2): GraphConv(channel_in=128, channel_out=128, n_edge_type=7, avg_degree=7, n_node_type=5)
      (skip_connection): Conv1x1(
        (linear): Linear(in_features=64, out_features=128, bias=False)

In [13]:
# split_recon = torch.ones((1,8,16,16,16))*(-1)
# split_recon[:,:,:,:,0:8] = batch[0].unsqueeze(0)
# split_recon = splitbig2split(samples)
# split_recon = splitbig2split(filled)
from octree_diff.octree.util_dualoctree import *
split_big = split2splitbig(split_recon)
split_recon = splitbig2split(split_big)
octree_recon = split2octree_small(split_recon, 6, 4)
octree_in = octree_recon
octree_out = vae.create_child_octree(octree_in).cuda()
doctree = DualOctree(octree_out)
doctree.post_processing_for_docnn()
z_rand = torch.randn(doctree.total_num, LATENT_DIM)
z_T = z_rand.cuda()


z_sampled = ddim_sample_logsnr_cosine(
    z_T=z_T,
    model=unet_model,
    doctree=doctree,
    S=50
    
)


output = vae.decode_code(z_sampled.cuda(), doctree, update_octree=False, pos=None)
recon_voxel = reconstruct_voxel_from_patch(output['sem_voxs'], octree_recon.cuda(), depth=6, shape=(1, 256, 256, 32), patch_size=4)

visualize_kitti_instance(recon_voxel[0])

<env:octfusion>/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
<kitti-weights>/util_octree_stuff.py:539: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
import torch
import torch.nn.functional as F

# --- helper: q(x_t | x0) ---
def q_sample(x0, t, sched, noise=None):
    """
    x0:    [B,C,D,H,W]
    t:     [B] int
    noise: [B,C,D,H,W] or None
    """
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_ab = _extract(sched["sqrt_alphas_cumprod"], t, x0.shape)
    sqrt_omb = _extract(sched["sqrt_one_minus_alphas_cumprod"], t, x0.shape)
    return sqrt_ab * x0 + sqrt_omb * noise

def _maybe_broadcast_mask(mask, x_like):
    """
    mask: [B,1,D,H,W] or [B,C,D,H,W] (float in [0,1])
    returns: [B,C,D,H,W]
    """
    if mask.dim() != x_like.dim():
        raise ValueError(f"mask dim {mask.dim()} must match x dim {x_like.dim()}")
    if mask.shape[1] == 1 and x_like.shape[1] != 1:
        mask = mask.expand(-1, x_like.shape[1], -1, -1, -1)
    return mask

def _feather_mask(mask, ks):
    """Box-blur mask with odd kernel size ks (approx Gaussian)."""
    if ks <= 1:
        return mask
    pad = ks // 2
    # avg_pool3d expects stride=1 + same padding to keep shape
    return F.avg_pool3d(mask, kernel_size=ks, stride=1, padding=pad)

def q_forward_xt_to(xs, t_from, t_to, sched, noise=None, eps=1e-7):
    """
    Multi-step forward diffusion: q(x_{s} | x_{t}) with s>t using ᾱ-ratio.
    xs:    [B,C,D,H,W] at time t_from
    t_from,t_to: [B] longs with t_to >= t_from
    """
    if noise is None:
        noise = torch.randn_like(xs)
    # sqrt ratio = sqrt(ᾱ_{to}) / sqrt(ᾱ_{from})
    sqrt_ab_from = _extract(sched["sqrt_alphas_cumprod"], t_from, xs.shape)
    sqrt_ab_to   = _extract(sched["sqrt_alphas_cumprod"], t_to,   xs.shape)
    sqrt_ratio   = (sqrt_ab_to / (sqrt_ab_from + eps)).clamp(0.0, 1.0)
    # ensure numeric safety
    add_var = (1.0 - sqrt_ratio**2).clamp_min(0.0)
    return sqrt_ratio * xs + add_var.sqrt() * noise
@torch.no_grad()
def sample_ddpm_inpaint_blended(
    model,
    sched,
    x_gt,                  # [B,C,D,H,W] clean ground truth
    mask,                  # [B,1,D,H,W] or [B,C,D,H,W], float {0,1} or [0,1]
    T=1000,
    device="cuda",
    x_T=None,              # optional starting noise [B,C,D,H,W]
    blend_pre=True,        # blend known region before model predicts (recommended)
    blend_post=True,       # blend known region after stepping to t-1 (recommended)
    feather_kernel_max=None,  # e.g., 7 -> large blur early, shrinks over time
):
    """
    DDPM ε-pred inpainting with 'blended diffusion' style clamping.
    - At each step t, we replace known-region with q(x_t | x_gt) (optionally feathered mask).
    - If feather_kernel_max is set (odd int), we use a larger blur early and smaller later.

    Returns: x_0 with known region exactly set to x_gt.
    """
    model.eval()
    x_gt = x_gt.to(device)
    mask = mask.to(device).clamp_(0.0, 1.0)
    mask = _maybe_broadcast_mask(mask, x_gt)

    B, C, D, H, W = x_gt.shape
    if x_T is None:
        x_t = torch.randn((B, C, D, H, W), device=device, dtype=x_gt.dtype)
    else:
        x_t = x_T.to(device, dtype=x_gt.dtype)

    # iterate t = T-1 ... 0
    for t_int in reversed(range(T)):
        
        t = torch.full((B,), t_int, device=device, dtype=torch.long)

        # Optional dynamic feathering (soft mask wider early, tighter later)
        if feather_kernel_max is not None and feather_kernel_max > 1:
            # scale kernel ~ linearly with time; ensure odd >=1
            ks = int(max(1, round(1 + (feather_kernel_max - 1) * (t_int / max(1, T - 1)))))
            if ks % 2 == 0:
                ks += 1
            mask_t = _feather_mask(mask, ks)
        else:
            mask_t = mask

        # Compute the *noised* ground-truth for this t to clamp the known region
        eps_known = torch.randn_like(x_t)
        x_gt_t = q_sample(x_gt, t, sched, noise=eps_known)

        # Pre-blend: ensure model sees correct known-region statistics at time t
        if blend_pre:
            x_t = mask_t * x_gt_t + (1.0 - mask_t) * x_t

        # Predict posterior using ε-pred
        mean, log_var, x0_pred = p_mean_variance_eps(model, x_t, t, sched)

        if t_int > 0:
            noise = torch.randn_like(x_t)
            x_t = mean + torch.exp(0.5 * log_var) * noise

            # Post-blend into x_{t-1}: use t-1 for the clamped known region
            if blend_post:
                t_prev = torch.full((B,), t_int - 1, device=device, dtype=torch.long)
                x_gt_tminus1 = q_sample(x_gt, t_prev, sched)  # fresh noise; ok for known region
                # Feather again at t-1 for smooth annealing
                if feather_kernel_max is not None and feather_kernel_max > 1:
                    ks = int(max(1, round(1 + (feather_kernel_max - 1) * ((t_int - 1) / max(1, T - 1)))))
                    if ks % 2 == 0:
                        ks += 1
                    mask_tminus1 = _feather_mask(mask, ks)
                else:
                    mask_tminus1 = mask
                x_t = mask_tminus1 * x_gt_tminus1 + (1.0 - mask_tminus1) * x_t
        else:
            # Final: return x_0 but *exactly* enforce known region = x_gt
            x_t = mask * x_gt + (1.0 - mask) * x0_pred

    return x_t

def _build_repaint_times(T, jump_length=10, jump_n_sample=10):
    """
    Create a schedule of timesteps t that moves by ±1 each transition,
    inserting zig-zag resampling blocks (forward r, backward r) at multiples of jump_length.
    """
    times = [T - 1]
    t = T - 1
    while t > 0:
        # one normal reverse step
        times.append(t - 1)
        # insert resampling block at multiples of jump_length
        if (t % max(1, jump_length) == 0) and (t > 0) and jump_length > 0 and jump_n_sample > 0:
            for _ in range(jump_n_sample):
                # r forward steps (increase time by +1)
                for _ in range(jump_length):
                    if times[-1] >= T - 1: break
                    times.append(times[-1] + 1)
                # r reverse steps (decrease time by -1)
                for _ in range(jump_length):
                    if times[-1] <= 0: break
                    times.append(times[-1] - 1)
        t -= 1
    return times

@torch.no_grad()
def sample_ddpm_inpaint_repaint(
    model,
    sched,
    x_gt,                  # [B,C,D,H,W]
    mask,                  # [B,1,...] or [B,C,...], float in [0,1]
    T=1000,
    device="cuda",
    x_T=None,
    jump_length=10,        # "j" in paper
    jump_n_sample=10,      # "n" in paper (how many zig-zags per anchor)
    blend_pre=False,       # if True: show model a clamped x_t (not required by RePaint)
):
    """
    Faithful RePaint (Lugmayr et al., 2022):
      - Reverse steps (t->t-1): unknown from model; known from q(x_{t-1}|x0_known); mask-combine (Eq. 8).
      - Forward steps (t->t+1): q(x_{t+1}|x_t) on the *current* sample; no clamping.
      - Time schedule is zig-zag per Fig. 9/10 with (jump_length, jump_n_sample).
    """
    model.eval()
    x_gt = x_gt.to(device)
    mask = _maybe_broadcast_mask(mask.to(device).clamp_(0, 1), x_gt)

    B, C, D, H, W = x_gt.shape
    x = torch.randn((B, C, D, H, W), device=device, dtype=x_gt.dtype) if x_T is None else x_T.to(device, dtype=x_gt.dtype).contiguous()

    # precompute schedule of times; iterate pairwise (t_last -> t_cur), each differs by ±1
    times = _build_repaint_times(T, jump_length=jump_length, jump_n_sample=jump_n_sample)
    # sanity: keep within [0, T-1]
    times = [int(max(0, min(T - 1, t))) for t in times]

    for t_last, t_cur in zip(times[:-1], times[1:]):
        t_last_b = torch.full((B,), t_last, device=device, dtype=torch.long)
        t_cur_b  = torch.full((B,), t_cur,  device=device, dtype=torch.long)

        # (Optional) pre-blend so the model "sees" correct known stats at t_last
        if blend_pre and t_cur < t_last:
            x_known_tlast = q_sample(x_gt, t_last_b, sched)
            x = mask * x_known_tlast + (1.0 - mask) * x

        if t_cur < t_last:
            # ------ Reverse diffusion: one step t_last -> t_cur (= t_last-1) ------
            mean, log_var, x0_pred = p_mean_variance_eps(model, x, t_last_b, sched)

            if t_cur > 0:
                # sample x_{t-1} from posterior
                noise = torch.randn_like(x)
                x_tm1 = mean + torch.exp(0.5 * log_var) * noise
            else:
                # last step: deterministic to x0_pred
                x_tm1 = x0_pred

            # sample known region at t_cur from given image and combine (Eq. 8)
            x_known_tm1 = q_sample(x_gt, t_cur_b, sched)  # q(x_{t-1} | x0_known)
            x = mask * x_known_tm1 + (1.0 - mask) * x_tm1

        else:
            # ------ Forward diffusion: one step t_last -> t_cur (= t_last+1) ------
            noise = torch.randn_like(x)
            x = q_forward_xt_to(x, t_last_b, t_cur_b, sched, noise=noise)

        # guards (helpful against cuBLAS/cuDNN failures if NaNs creep in)
        if torch.isnan(x).any() or torch.isinf(x).any():
            raise RuntimeError(f"NaN/Inf detected after transition {t_last}->{t_cur}")

    # ensure exact known region at x_0
    return mask * x_gt + (1.0 - mask) * x


In [17]:
x_gt = torch.zeros(1,1,32,32,16).cuda()
x_gt[:,:,0:16,:,:] = samples[:,:,16:,:,:]
mask = torch.zeros(1,1,32,32,16).cuda()
mask[:,:,0:16,:,:] = 1


In [18]:
device = next(structure_model.parameters()).device
sched = build_ddpm_sampling_buffers(T=1000, device=device, beta_start=1e-4, beta_end=2e-2)

# x_gt: [B,C,D,H,W], mask: [B,1,D,H,W] (1=keep, 0=generate)
with torch.inference_mode():
    # Use EMA if you have it
    # with use_ema_weights(structure_model, ema):
    x0_inpaint = sample_ddpm_inpaint_blended(
        model=structure_model,
        sched=sched,
        x_gt=x_gt,
        mask=mask,
        T=1000,
        device=device,
        x_T=None,
        blend_pre=True,
        blend_post=True,
        feather_kernel_max=None,   # try 5–9 for soft boundaries; or None for hard mask
    )


In [19]:
samples_clone.shape

torch.Size([1, 1, 32, 32, 16])

In [20]:
split_big = torch.zeros(1,1,32,32,32).cuda()
split_big[:,:,:,:,0:16] = samples_clone
split_recon = splitbig2split(split_big)
octree_recon = split2octree_small(split_recon, 6, 4)

visualize_kitti_instance(samples_clone[0][0])

In [21]:
samples_clone_outpaint = x0_inpaint.clone()
samples_clone_outpaint[samples_clone_outpaint>0] = 1
samples_clone_outpaint[samples_clone_outpaint<0] = 0
visualize_kitti_instance(samples_clone_outpaint[0][0])

split_recon_outpaint = torch.zeros((1,1,32,32,32))
split_recon_outpaint[:,:,:,:,0:16] = samples_clone_outpaint

split_recon_outpaint = splitbig2split(split_recon_outpaint)
octree_recon_outpaint = split2octree_small(split_recon_outpaint, 6, 4)




In [22]:
samples_clone.shape

torch.Size([1, 1, 32, 32, 16])

In [23]:
vox_combine = torch.zeros((48,32,16)).cuda()
vox_combine[0:32,:,:] = samples_clone[0][0]
vox_combine[32:48,:,:]  = samples_clone_outpaint[0][0][16:32,:,:]
visualize_kitti_instance(vox_combine)

# inpaint sem

In [24]:
octree_in_recon = octree_recon.cuda()
octree_out_recon = vae.create_child_octree(octree_in_recon).cuda()
doctree_recon = DualOctree(octree_out_recon)
doctree_recon.post_processing_for_docnn()
z_rand = torch.randn(doctree_recon.total_num, LATENT_DIM)
z_T = z_rand.cuda()


z_sampled = ddim_sample_logsnr_cosine(
    z_T=z_T,
    model=unet_model,
    doctree=doctree_recon,
    S=50
    
)


output = vae.decode_code(z_sampled.cuda(), doctree_recon, update_octree=False, pos=None)
recon_voxel = reconstruct_voxel_from_patch(output['sem_voxs'], octree_recon.cuda(), depth=6, shape=(1, 256, 256, 32), patch_size=4)

visualize_kitti_instance(recon_voxel[0])

In [25]:
octree_in_recon_outpaint = octree_recon_outpaint.cuda()
octree_out_recon_outpaint = vae.create_child_octree(octree_in_recon_outpaint).cuda()
doctree_recon_outpaint = DualOctree(octree_out_recon_outpaint)
doctree_recon_outpaint.post_processing_for_docnn()
z_rand_outpaint = torch.randn(doctree_recon_outpaint.total_num, LATENT_DIM)
z_T_outpaint = z_rand_outpaint.cuda()


z_sampled_outpaint = ddim_sample_logsnr_cosine(
    z_T=z_T_outpaint,
    model=unet_model,
    doctree=doctree_recon_outpaint,
    S=  50
    
)


output = vae.decode_code(z_sampled_outpaint.cuda(), doctree_recon_outpaint, update_octree=False, pos=None)
recon_voxel2 = reconstruct_voxel_from_patch(output['sem_voxs'], octree_recon_outpaint.cuda(), depth=6, shape=(1, 256, 256, 32), patch_size=4)

visualize_kitti_instance(recon_voxel2[0])

In [26]:
offset_recon = doctree_recon.total_num - doctree_recon.nnum[6]
offset_recon_outpaint = doctree_recon_outpaint.total_num - doctree_recon_outpaint.nnum[6]
offset_recon,offset_recon_outpaint

(tensor(6126, dtype=torch.int32), tensor(6193, dtype=torch.int32))

In [27]:
# # 2 octrees
z_gt = torch.zeros(z_T_outpaint.shape).cuda()
mask = torch.zeros(z_T_outpaint.shape).cuda()


# Generate all voxel coordinates in the specified range
coords = torch.stack(torch.meshgrid(
    torch.arange(32, 64),     # i
    torch.arange(0, 64),      # j
    torch.arange(0, 32),      # k
    indexing='ij'
), dim=-1).reshape(-1, 3)

# Add the batch index (assumed to be 0 as in your original code)
xyzb = torch.cat([coords, torch.zeros(coords.shape[0], 1, dtype=torch.int32)], dim=1).cuda()

# Search in both octrees
idx = octree_in_recon.search_xyzb(xyzb, depth=6)


coords_dst = torch.stack(torch.meshgrid(
    torch.arange(0, 32),     # i
    torch.arange(0, 64),      # j
    torch.arange(0, 32),      # k
    indexing='ij'
), dim=-1).reshape(-1, 3)
xyzb_dst = torch.cat([coords_dst, torch.zeros(coords_dst.shape[0], 1, dtype=torch.int32)], dim=1).cuda()

idx_outpaint = octree_in_recon_outpaint.search_xyzb(xyzb_dst, depth=6)

# Filter out invalid (-1) indices
valid = (idx != -1) & (idx_outpaint != -1)

# Apply offsets and update z_gt and mask
z_gt[idx_outpaint[valid] + offset_recon_outpaint] = z_sampled[idx[valid] + offset_recon]
mask[idx_outpaint[valid] + offset_recon_outpaint] = 1


In [28]:
(idx!=-1).sum(), (idx_outpaint!=-1).sum()

(tensor(12664, device='cuda:0'), tensor(12664, device='cuda:0'))

In [34]:
z_T = torch.randn_like(z_gt)
@torch.no_grad()
def ddim_sample_logsnr_cosine_inpaint(z_T,z_gt, model,mask, doctree, S=50, jump_n_sample = 10,predict_mode="x0"):
    """
    DDIM sampling using cosine logSNR schedule and x₀ prediction.

    Args:
        z_T: Initial noise tensor [N, C]
        model: Trained diffusion model
        doctree: DualOctree structure
        S: Number of DDIM steps (e.g., 50)
        predict_mode: "x0" (only supported here)
    """
    assert predict_mode == "x0", "Only x0 mode is supported in this version."

    # === Step 1: Generate continuous t schedule [1.0, ..., 0.0]
    t_steps = torch.linspace(1., 0., S + 1, device=z_T.device)  # [S+1]
    t_pairs = list(zip(t_steps[:-1], t_steps[1:]))


    z_t = z_T
    for t, t_next in t_pairs:
        # Convert scalars to tensors
        for j in range(jump_n_sample): 
            t_tensor = t.view(1, 1)
            t_next_tensor = t_next.view(1, 1)

            logsnr_t = alpha_cosine_log_snr(t_tensor)
            logsnr_next = alpha_cosine_log_snr(t_next_tensor)

            alpha_t, sigma_t = log_snr_to_alpha_sigma(logsnr_t)
            alpha_next, sigma_next = log_snr_to_alpha_sigma(logsnr_next)
            # Inpaint
            noised_z_gt = (z_gt * alpha_next + sigma_next * torch.randn_like(z_gt))
            x0_pred = model(z_t, doctree=doctree, timesteps=t_tensor)  # Predict x0
            x0_pred = ( mask) * noised_z_gt + (1-mask) * x0_pred
            # DDIM update rule
            z_t = alpha_next * x0_pred + sigma_next * torch.randn_like(z_t)
            if j < jump_n_sample - 1 and t > 1:
                noise_back = torch.randn_like(z_t)
                # noised_data = alpha_t * noised_data + sigma_t * noise_back
                z_t = alpha_t * z_t + sigma_t * noise_back
    return z_t
z_sampled_outpaint = ddim_sample_logsnr_cosine_inpaint(z_T,z_gt, unet_model,mask, doctree_recon_outpaint, S=50,jump_n_sample=5, predict_mode="x0")

In [35]:
output_outpaint = vae.decode_code(z_sampled_outpaint.cuda(), doctree_recon_outpaint, update_octree=False, pos=None)
recon_voxel_outpaint = reconstruct_voxel_from_patch(output_outpaint['sem_voxs'], octree_recon_outpaint.cuda(), depth=6, shape=(1, 256, 256, 32), patch_size=4)



In [36]:
voxel_combined = torch.zeros((1,384,256,32))
voxel_combined[:,0:128,:,:] = recon_voxel[:,0:128,:,:]
voxel_combined[:,128:384,:,:] = recon_voxel_outpaint[:,:,:,:]

In [37]:
visualize_kitti_instance(recon_voxel[0])

In [45]:
visualize_kitti_instance(voxel_combined[0])

In [46]:
import os
import numpy as np
import torch
i = 11
# === Config ===
save_dir = "vis_dataset/sequences/03/predictions"
os.makedirs(save_dir, exist_ok=True)

# === Prepare lookup table for remapping once (fast) ===
custom_to_semkitti = {
    0:  0,   1: 10,  2: 11,  3: 15,  4: 18,
    5: 20,   6: 30,  7: 31,  8: 32,  9: 40,
    10: 44, 11: 48, 12: 49, 13: 50, 14: 51,
    15: 70, 16: 71, 17: 72, 18: 80, 19: 81,
    20: 0, 255: 0,
}
lookup = np.zeros(256, dtype=np.uint16)
for k, v in custom_to_semkitti.items():
    lookup[k] = v


pred_np = voxel_combined[0].cpu().numpy().astype(np.uint8)
remapped = lookup[pred_np]  # vectorized remapping to SemanticKITTI IDs

# === Save to .label ===
save_path = os.path.join(save_dir, f"{i:06d}.label")
remapped.tofile(save_path)

print(f"[INFO] Saved prediction for frame {i:06d} to {save_path}")

print("[DONE] All predictions saved to:", save_dir)


[INFO] Saved prediction for frame 000011 to vis_dataset/sequences/03/predictions/000011.label
[DONE] All predictions saved to: vis_dataset/sequences/03/predictions


In [43]:
remapped.shape

(384, 256, 32)

# whole plane

In [37]:
x1_clone = x1.clone()
x1_clone[x1_clone < 0] = 0
x1_clone[x1_clone > 0] = 1
visualize_kitti_instance(x1_clone[0][0])

NameError: name 'x1' is not defined

In [117]:
x2_clone = x2.clone()
x2_clone[x2_clone < 0] = 0
x2_clone[x2_clone > 0] = 1
visualize_kitti_instance(x2_clone[0][0])

In [118]:
x3_clone = x3.clone()
x3_clone[x3_clone < 0] = 0
x3_clone[x3_clone > 0] = 1
visualize_kitti_instance(x3_clone[0][0])

In [42]:
import torch

import torch

# --- your order (unchanged) ---
def spiral_centers_halfstep(n):
    x, y = 0, 0
    path = [(0,0)]
    for i in range(1, n+1):
        if i % 2 == 1:
            x += 1; path.append((x,y))
            for _ in range(i):
                y += 1; path.append((x,y))
            for _ in range(i):
                x -= 1; path.append((x,y))
        else:
            x -= 1; path.append((x,y))
            for _ in range(i):
                y -= 1; path.append((x,y))
            for _ in range(i):
                x += 1; path.append((x,y))
    return path

def build_mask_by_neighbors(tx, ty, visited, *, tile=32, depth=16,
                            device="cuda", dtype=torch.float32):
    L = tile
    known = torch.zeros((L, L), dtype=torch.bool, device=device)

    # left neighbor → upper rows known
    if (tx-1, ty) in visited:
        known[:L//2, :] = True

    # right neighbor → lower rows known
    if (tx+1, ty) in visited:
        known[L//2:, :] = True

    # above neighbor → left columns known
    if (tx, ty-1) in visited:
        known[:, :L//2] = True

    # below neighbor → right columns known
    if (tx, ty+1) in visited:
        known[:, L//2:] = True

    # Final mask shape: [B,1,H,W,D]
    mask = torch.zeros((1, 1, L, L, depth), device=device, dtype=dtype)
    mask[:, :, :, :, :] = known[None, None, :, :, None].to(dtype)
    return mask, known











def vis_slice(mask, depth_idx=0):
    """
    Visualize a slice of mask [1,1,32,32,16] in Jupyter.
    - First coord (axis=2) = horizontal (x-axis).
    - Second coord (axis=3) = vertical (y-axis).
    - Slice along axis=4 (depth).
    0 → white, 1 → black.
    """
    assert mask.shape == (1,1,32,32,16), f"Expected (1,1,32,32,16), got {mask.shape}"
    
    # extract slice [X,Y] at given depth
    slice2d = mask[0,0,:,:,depth_idx].detach().cpu().numpy()
    
    # transpose to put X on horizontal, Y on vertical
    img = np.where(slice2d == 0, 1.0, 0.0).T  
    
    # flip vertical axis so "bottom half" shows at bottom
    img = np.flipud(img)
    
    # upscale with 2x2 blocks
    img_big = np.kron(img, np.ones((2,2)))
    
    plt.figure(figsize=(4,4))
    plt.imshow(img_big, cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    plt.show()

# Example:
# vis_slice(mask, depth_idx=0)

def outpaint_square_spiral(structure_model, sched, L, device):
    tile, stride, depth = 32, 16, 16
    H = W = tile + (L-1)*stride
    canvas = torch.zeros((1, H, W, depth), device=device)

    steps = spiral_centers_halfstep(L-1)
    min_x = min(tx for tx,ty in steps)
    min_y = min(ty for tx,ty in steps)
    visited = set()

    for i, (tx,ty) in enumerate(steps):
        # pixel coords for top-left of tile
        x0 = stride*(tx - min_x)
        y0 = stride*(ty - min_y)

        print(f"\nStep {i}: canvas {H}×{W}×{depth}, tile center=({tx},{ty}), top-left=({x0},{y0})")

        # Build mask from neighbors in tile coordinates
        mask, known2d = build_mask_by_neighbors(tx, ty, visited,
                                                tile=tile, depth=depth, device=device)

        vis_slice(mask)
        # Build x_gt for this tile
        x_gt = canvas[ :,x0:x0+tile, y0:y0+tile, :].clone()

        x_gt = x_gt.unsqueeze(1) 
        # Sample
        x_T = torch.randn((1, 1, tile, tile, depth), device=device)

        # out = sample_ddpm_inpaint_blended(
        #     model=structure_model, sched=sched,
        #     x_gt=x_gt, mask=mask,
        #     T=1000, device=device, x_T=x_T,
        #     blend_pre=True, blend_post=True, feather_kernel_max=0,
                # )
        out = sample_ddpm_inpaint_repaint(
            model=structure_model,
            sched=sched,
            x_gt=x_gt,
            mask=mask,
            T=1000,
            device=device,
            x_T=x_T,
            jump_length=5,     # "j" 
            jump_n_sample=5,   # "n" 
  
            blend_pre=True     
        )
        # Update canvas (convert out back to [1,H,W,D] layout)
        out_tile = out.squeeze(1)
        canvas[:, x0:x0+tile, y0:y0+tile, :] = out_tile

        visited.add((tx,ty))
        # if i == 2:
        #     break
    return canvas, out,out_tile ,x_gt,mask


In [43]:
canvas,out,out_tile ,xgt,mask = outpaint_square_spiral(structure_model,sched,3,device)


Step 0: canvas 64×64×16, tile center=(0,0), top-left=(16,16)


/tmp/ipykernel_324581/105949970.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Step 1: canvas 64×64×16, tile center=(1,0), top-left=(32,16)

Step 2: canvas 64×64×16, tile center=(1,1), top-left=(32,32)

Step 3: canvas 64×64×16, tile center=(0,1), top-left=(16,32)

Step 4: canvas 64×64×16, tile center=(-1,1), top-left=(0,32)

Step 5: canvas 64×64×16, tile center=(-1,0), top-left=(0,16)

Step 6: canvas 64×64×16, tile center=(-1,-1), top-left=(0,0)

Step 7: canvas 64×64×16, tile center=(0,-1), top-left=(16,0)

Step 8: canvas 64×64×16, tile center=(1,-1), top-left=(32,0)


In [44]:
def visualize_structure(voxels):
    voxels_clone = voxels.clone()
    voxels_clone[voxels_clone < 0] = 0
    voxels_clone[voxels_clone > 0] = 1
    visualize_kitti_instance(voxels_clone[0])   

In [45]:
visualize_structure(canvas)

In [32]:

canvas.shape

torch.Size([1, 96, 96, 16])

In [50]:
torch.save(canvas, f"{KITTI_WEIGHTS}/canvas3.pt")

# the outpainting sem

canvas is 96 96 16.
then for each 32 32 16( overlaps, 25 in total)
padd to 32 32 32
then convert to split signal, then to 25 octree.
then do the inpaint. 
the latent should be 64*3 64*3 64 
so first the sem canvas is 768 768 32. (96*8)







In [46]:
def construct_octree_dict(canvas, L, device):
    tile, stride, depth = 32, 16, 64

    steps = spiral_centers_halfstep(L-1)
    min_x = min(tx for tx,ty in steps)
    min_y = min(ty for tx,ty in steps)
    octree_dict = {}
    for i, (tx,ty) in enumerate(steps):
        # pixel coords for top-left of tile
        x0 = stride*(tx - min_x)
        y0 = stride*(ty - min_y)
        split_big = torch.zeros(1,1,32,32,32).cuda()
        split_big[:,:,:,:,0:16] = canvas[:,x0:x0+32,y0:y0+32,:]
        octree = split2octree_small(splitbig2split(split_big), 6, 4)
        octree_dict[tx,ty] = octree
    return octree_dict
        


In [47]:
octree_dict = construct_octree_dict(canvas, 3,device)

In [20]:
for key, val in octree_dict.items():
    print(key, val)
import torch

# ---- helpers ---------------------------------------------------------------

def _block_coords(i0,i1,j0,j1,k0,k1, device):
    # (i,j,k) with ij-indexing to match your example
    ii, jj, kk = torch.meshgrid(
        torch.arange(i0, i1, device=device),
        torch.arange(j0, j1, device=device),
        torch.arange(k0, k1, device=device),
        indexing='ij'
    )
    coords = torch.stack([ii, jj, kk], dim=-1).reshape(-1, 3).to(torch.int32)
    return coords

@torch.no_grad()
def _copy_block_into(z_gt, mask, *,
                     dst_octree, dst_offset,
                     src_octree, src_offset,
                     src_box, dst_box,
                     batch_idx=0, depth_level=6):
    """
    Copy values from src half-block into dst half-block using octree index lookups.
    src_box/dst_box are (i0,i1,j0,j1,k0,k1) in *their respective* tile coordinate frames.
    """
    device = z_gt.device
    # Build coord list in src frame
    i0,i1,j0,j1,k0,k1 = src_box
    coords_src = _block_coords(i0,i1,j0,j1,k0,k1, device=device)

    # Attach batch index (matches your code)
    xyzb = torch.cat([coords_src, torch.full((coords_src.shape[0],1), batch_idx, dtype=torch.int32, device=device)], dim=1)

    # Search both octrees
    idx_src = src_octree.search_xyzb(xyzb, depth=depth_level)
    idx_dst = dst_octree.search_xyzb(xyzb, depth=depth_level)  # same spatial coords but in dst frame’s origin

    valid = (idx_src != -1) & (idx_dst != -1)
    z_src = output_dict[key]
    if valid.any():
        z_gt[idx_dst[valid] + dst_offset] = z_src[idx_src[valid] + src_offset]
        mask[idx_dst[valid] + dst_offset] = 1

# ---- main “define mask & gt” logic per tile --------------------------------
def get_offset(vae,octree):

    octree_recon = vae.create_child_octree(octree).cuda()
    doctree = DualOctree(octree_recon)
    offset = doctree.total_num - doctree.nnum[6]
    return offset
def define_mask_and_gt_for_tile(vae,current_key, octree_dict, output_dict,visited_keys,
                                *, I=64, J=64, K=32, device="cuda"):
    """
    Returns (z_gt, mask) for the current tile at `current_key=(x,y)`,
    filled from any visited neighbors (left/right/top/bottom).
    Shapes match your outpaint latent layout.
    """
    # Extract handles for current tile
    cur_octree = octree_dict[current_key]
   
    octree_out_recon = vae.create_child_octree(cur_octree).cuda()
    cur_doctree = DualOctree(octree_out_recon)
    cur_offset = get_offset(vae,cur_octree),
    # Preallocate
    z_rand_cur = torch.randn(cur_doctree.total_num, LATENT_DIM)
    z_gt  = torch.zeros_like(z_rand_cur, device=device)   # same flat storage type/shape
    mask  = torch.zeros_like(z_rand_cur, device=device)

    # neighbor keys
    x,y = current_key
    left_key   = (x-1, y)
    right_key  = (x+1, y)
    top_key    = (x,   y+1)
    bottom_key = (x,   y-1)

    # LEFT neighbor: copy its RIGHT half -> current LEFT half
    if left_key in visited_keys:
        nbr_octree = octree_dict[left_key]
        _copy_block_into(
            z_gt, mask,
            dst_octree=cur_doctree,  dst_offset=cur_offset ,
            src_octree=nbr_octree, src_offset=get_offset(vae,nbr_octree),
            src_box=(I//2, I, 0, J, 0, K),     # right half of LEFT neighbor
            dst_box=(0, I//2, 0, J, 0, K),     # left half of CURRENT
        )

    # RIGHT neighbor: copy its LEFT half -> current RIGHT half
    if right_key in visited_keys:
        nbr_octree = octree_dict[right_key]
        _copy_block_into(
            z_gt, mask,
            dst_octree=cur_doctree,  dst_offset=cur_offset ,
            src_octree=nbr_octree, src_offset=get_offset(vae,nbr_octree),
            src_box=(0, I//2, 0, J, 0, K),     # left half of RIGHT neighbor
            dst_box=(I//2, I, 0, J, 0, K),     # right half of CURRENT
        )

    # TOP neighbor: copy its BOTTOM half -> current TOP half
    if top_key in visited_keys:
        nbr_octree = octree_dict[top_key]
        _copy_block_into(
            z_gt, mask,
            dst_octree=cur_doctree,  dst_offset=cur_offset ,
            src_octree=nbr_octree, src_offset=get_offset(vae,nbr_octree),
            src_box=(0, I, J//2, J, 0, K),     # bottom half of TOP neighbor
            dst_box=(0, I, 0, J//2, 0, K),     # top half of CURRENT
        )

    # BOTTOM neighbor: copy its TOP half -> current BOTTOM half
    if bottom_key in visited_keys:
        nbr_octree = octree_dict[bottom_key]
        _copy_block_into(
            z_gt, mask,
            dst_octree=cur_doctree,  dst_offset=cur_offset ,
            src_octree=nbr_octree, src_offset=get_offset(vae,nbr_octree),
            src_box=(0, I, 0, J//2, 0, K),     # top half of BOTTOM neighbor
            dst_box=(0, I, J//2, J, 0, K),     # bottom half of CURRENT
        )

    return z_gt, mask


(0, 0) <ocnn.octree.octree.Octree object at 0x7f634074a1d0>
(1, 0) <ocnn.octree.octree.Octree object at 0x7f6339d04250>
(1, 1) <ocnn.octree.octree.Octree object at 0x7f63316320d0>
(0, 1) <ocnn.octree.octree.Octree object at 0x7f6331689550>
(-1, 1) <ocnn.octree.octree.Octree object at 0x7f6331630e50>
(-1, 0) <ocnn.octree.octree.Octree object at 0x7f6331638310>
(-1, -1) <ocnn.octree.octree.Octree object at 0x7f6341411b10>
(0, -1) <ocnn.octree.octree.Octree object at 0x7f6348a0afd0>
(1, -1) <ocnn.octree.octree.Octree object at 0x7f6331686e10>


In [73]:


def outpaint_latent_canvas(unet_model,vae, octree_dict, L, device):
    tile, stride, depth = 32, 16, 16
    L = 2*(tile + (L-1)*stride)
    output_dict = {}
    visited = set()
    for key, octree in octree_dict.items():
        x,y = key
        
        # define mask and gt
        
        
        
        z_gt, mask = define_mask_and_gt_for_tile(vae,key, octree_dict, visited,
                                             I=64, J=64, K=32, device=device)

        
        
        # inoutpaint
        output_dict[key] = sem_paint_octree(val, key[0], key[1])
   
        visited.add(key)
    return output_dict

In [62]:
octree_dict[0,0]

In [21]:
def sem_paint_octree(octree_dict,x,y):
    
    octree_in_recon_outpaint = octree_dict[x,y].cuda()
    octree_out_recon_outpaint = vae.create_child_octree(octree_in_recon_outpaint).cuda()
    doctree_recon_outpaint = DualOctree(octree_out_recon_outpaint)
    doctree_recon_outpaint.post_processing_for_docnn()
    z_rand_outpaint = torch.randn(doctree_recon_outpaint.total_num, LATENT_DIM)
    z_T_outpaint = z_rand_outpaint.cuda()


    z_sampled_outpaint = ddim_sample_logsnr_cosine(
        z_T=z_T_outpaint,
        model=unet_model,
        doctree=doctree_recon_outpaint,
        S=  50
        
    )


    output = vae.decode_code(z_sampled_outpaint.cuda(), doctree_recon_outpaint, update_octree=False, pos=None)
    recon_voxel = reconstruct_voxel_from_patch(output['sem_voxs'], octree_dict[x,y].cuda(), depth=6, shape=(1, 256, 256, 32), patch_size=4)

    # visualize_kitti_instance(recon_voxel[0])
    return recon_voxel[0]

In [64]:
sem_canvas = torch.zeros((256*3,256*3,32)).cuda()
for x in range(-2,4,2):
    for y in range(-2,4,2):
        sem_canvas[(x+2)*128:(x+4)*128,(y+2)*128:(y+4)*128,:] = sem_paint_octree(octree_dict,x,y)

In [48]:
sem_canvas = torch.zeros((256*3,256*3,32)).cuda()
for x in range(-1,1,1):
    for y in range(-1,1,1):
        sem_canvas[(x+1)*256:(x+2)*256,(y+1)*256:(y+2)*256,:] = sem_paint_octree(octree_dict,x,y)

<env:octfusion>/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


In [51]:
visualize_kitti_instance(sem_canvas)

In [79]:
for x in range(-1,1,1):
    for y in range(-1,1,1):
        print(x)
        print((x+1)*256,(x+2)*256)

-1
0 256
-1
0 256
0
256 512
0
256 512
